In [4]:
import pandas as pd
import joblib
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Load data
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

# Load preprocessor
preprocessor = joblib.load('../models/preprocessor.joblib')

In [5]:
lr_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)

print(f"Linear Regression MAE: {mean_absolute_error(y_test, lr_preds):.2f}")
print(f"Linear Regression R2: {r2_score(y_test, lr_preds):.2f}")

Linear Regression MAE: 4233.47
Linear Regression R2: 0.79


In [6]:
gb_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))
])

gb_pipeline.fit(X_train, y_train)
gb_preds = gb_pipeline.predict(X_test)

print(f"Gradient Boosting MAE: {mean_absolute_error(y_test, gb_preds):.2f}")
print(f"Gradient Boosting R2: {r2_score(y_test, gb_preds):.2f}")

# Save the best model
joblib.dump(gb_pipeline, '../models/gb_model.joblib')

Gradient Boosting MAE: 2450.41
Gradient Boosting R2: 0.88


['../models/gb_model.joblib']

In [7]:
# Create a results dataframe
results = X_test.copy()
results['actual'] = y_test
results['predicted'] = gb_preds
results['error'] = (results['actual'] - results['predicted']).abs()

# Group by segments to find where MAE is highest
segment_analysis = results.groupby(['age_band', 'bmi_band'], observed=True)['error'].agg(['mean', 'count']).rename(columns={'mean': 'MAE'})
segment_analysis = segment_analysis.sort_values(by='MAE', ascending=False)

print("Top 5 Segments with Highest Error (MAE):")
print(segment_analysis.head())

Top 5 Segments with Highest Error (MAE):
                                 MAE  count
age_band bmi_band                          
31-45    normal          5203.843796      4
60+      severely_obese  3395.871833      8
18-30    obese           3116.039097     28
46-60    severely_obese  3062.655728     22
31-45    severely_obese  3053.006512     25
